Cortex AI enrichment pipeline on federated Iceberg tables with Horizon governance


# Scenario 2 — Cortex AI Enrichment Pipeline on Iceberg

**What this demonstrates:**
- Snowflake Cortex LLM functions enrich federated Iceberg data
- AI-generated results persist as a new Snowflake-managed Iceberg table
- Horizon governance applies to AI output the same way it applies to raw data
- A Cortex Agent queries all three Iceberg tables using natural language

**Prerequisites:**
- Run `01_sf_iceberg_catalog_setup.sql` (Scenario 1 tables)
- Run `01_sf_iceberg_catalog_setup.sql`
- Warehouse `LOAD_WH` available

In [ ]:
%%sql -r session_setup
-- Session configuration
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE IDENTIFIER($SF_WAREHOUSE);
USE DATABASE HORIZON_DEMO_SFDB;
USE SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r variables_set
-- Parameters — change only SF_WAREHOUSE if your warehouse differs
SET SF_WAREHOUSE          = 'LOAD_WH';         -- !! REPLACE if different
SET SF_READER_ROLE        = 'EXT_COMPUTE_ENG_DEMO_ROLE';
-- Pre-set demo values
SET SF_MANAGED_ICEBERG_DB = 'HORIZON_DEMO_SFDB';
SET SF_DEMO_SCHEMA        = 'DEMO_SCHEMA';
SET SF_FEDERATED_DB       = 'DATABRICKS_DEMO_DB';
SET SF_FEDERATED_SCHEMA   = 'horizon_demo';
SET SF_EXTERNAL_VOLUME    = 'SNOWFLAKE_MANAGED';

SET AI_INSIGHTS_TBL  = $SF_MANAGED_ICEBERG_DB || '.' || $SF_DEMO_SCHEMA || '.AI_ORDER_INSIGHTS';
SET FED_ORDERS       = $SF_FEDERATED_DB || '.' || $SF_FEDERATED_SCHEMA || '.customer_orders';
SET FED_SENSITIVE    = $SF_FEDERATED_DB || '.' || $SF_FEDERATED_SCHEMA || '.sensitive_orders';
SET MASK_RISK        = $SF_MANAGED_ICEBERG_DB || '.' || $SF_DEMO_SCHEMA || '.MASK_RISK_LEVEL';
SET SEMANTIC_VIEW    = $SF_MANAGED_ICEBERG_DB || '.' || $SF_DEMO_SCHEMA || '.ICEBERG_AI_SEMANTIC_VIEW';

In [ ]:
%%sql -r cortex_check
-- Verify Cortex LLM functions are available in this region
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-8b',
    'Reply with exactly one word: ready'
) AS cortex_status;

## Step 2 — Create AI_ORDER_INSIGHTS Iceberg Table

Three-phase approach to avoid internal error `370001`:
- Cortex LLM functions inside CTAS that targets an Iceberg table trigger an internal error when the source is a federated CLD table
- **Fix:** (a) stage federated data in a temp table, (b) create the Iceberg table from local data WITHOUT Cortex, (c) UPDATE the Iceberg table with Cortex enrichment separately

In [ ]:
-- 2a: Stage federated data locally (plain CTAS from CLD into temp table)
SET TMP_STAGED_ORDERS = $SF_MANAGED_ICEBERG_DB || '.' || $SF_DEMO_SCHEMA || '.TMP_STAGED_ORDERS';
CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($TMP_STAGED_ORDERS)
AS
WITH deduped_orders AS (
    SELECT order_id, customer_id, product, amount, order_date, status
    FROM IDENTIFIER($FED_ORDERS)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_id) = 1
),
deduped_sensitive AS (
    SELECT order_id, region
    FROM IDENTIFIER($FED_SENSITIVE)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_id) = 1
)
SELECT
    co.order_id,
    co.customer_id,
    co.product,
    co.amount,
    co.order_date,
    co.status,
    COALESCE(so.region, 'UNKNOWN') AS region,
    CASE
        WHEN co.status = 'CANCELLED' THEN 'HIGH'
        WHEN co.amount >= 500        THEN 'HIGH'
        WHEN co.amount >= 100        THEN 'MEDIUM'
        ELSE 'LOW'
    END AS risk_level
FROM deduped_orders co
LEFT JOIN deduped_sensitive so ON co.order_id = so.order_id;

In [ ]:
-- 2b: Create Iceberg table from local temp data (no Cortex call here)
CREATE OR REPLACE ICEBERG TABLE IDENTIFIER($AI_INSIGHTS_TBL)
    CATALOG = 'SNOWFLAKE'
    AS
SELECT
    order_id,
    customer_id,
    product,
    amount,
    order_date,
    status,
    region,
    risk_level,
    ''::VARCHAR AS ops_note,
    CURRENT_TIMESTAMP()::TIMESTAMP_LTZ(6) AS enriched_at
FROM IDENTIFIER($TMP_STAGED_ORDERS);

In [ ]:
%%sql -r cortex_enrich
-- 2c: Enrich with Cortex (UPDATE on local Iceberg table — no CLD involved)
UPDATE IDENTIFIER($AI_INSIGHTS_TBL) t
SET ops_note = SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-8b',
    CONCAT(
        'You are a logistics AI assistant. Write one actionable sentence (max 15 words) ',
        'for the operations team about this order. ',
        'Product: ', t.product, ', Amount: $', t.amount::STRING,
        ', Status: ', t.status,
        CASE WHEN t.region <> 'UNKNOWN' THEN ', Region: ' || t.region ELSE '' END, '.'
    )
)::VARCHAR;

In [ ]:
-- 2d: Clean up temp table
DROP TABLE IF EXISTS IDENTIFIER($TMP_STAGED_ORDERS);

## Step 3 — Apply Horizon Governance on AI Output

CREATE OR REPLACE TABLE (above) dropped the old table, releasing any masking policy associations. It is now safe to CREATE OR REPLACE the masking policy and re-apply it on every run.

In [ ]:
%%sql -r governance_applied
CREATE OR REPLACE MASKING POLICY IDENTIFIER($MASK_RISK)
    AS (val STRING)
    RETURNS STRING ->
    CASE
        WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN val
        ELSE '*** RESTRICTED ***'
    END;

ALTER ICEBERG TABLE IDENTIFIER($AI_INSIGHTS_TBL)
    MODIFY COLUMN risk_level
    SET MASKING POLICY IDENTIFIER($MASK_RISK);

## Step 4 — Create Semantic View for Cortex Agent

The semantic view allows a Cortex Agent to query both the AI-enriched Iceberg table and the original federated orders using natural language.

In [ ]:
-- Semantic view combines local Iceberg + both federated catalog-linked tables
USE WAREHOUSE IDENTIFIER($SF_WAREHOUSE);

CREATE OR REPLACE SEMANTIC VIEW HORIZON_DEMO_SFDB.DEMO_SCHEMA.ICEBERG_AI_SEMANTIC_VIEW
TABLES (
  orders     AS HORIZON_DEMO_SFDB.DEMO_SCHEMA.AI_ORDER_INSIGHTS
               PRIMARY KEY (order_id)
               WITH SYNONYMS = ('ai orders', 'enriched orders', 'risk orders'),

  fed_orders AS DATABRICKS_DEMO_DB.horizon_demo.customer_orders
               PRIMARY KEY (order_id)
               WITH SYNONYMS = ('federated orders', 'source orders', 'databricks orders'),

  fed_sensitive AS DATABRICKS_DEMO_DB.horizon_demo.sensitive_orders
               PRIMARY KEY (order_id)
               WITH SYNONYMS = ('sensitive orders', 'customer details', 'PII orders')
)
RELATIONSHIPS (
  orders(order_id) REFERENCES fed_orders,
  orders(order_id) REFERENCES fed_sensitive
)
FACTS (
  orders.amount AS amount WITH SYNONYMS = ('order value', 'revenue', 'price')
)
DIMENSIONS (
  orders.product       AS product       WITH SYNONYMS = ('item', 'product name'),
  orders.status        AS status        WITH SYNONYMS = ('fulfillment status', 'order status'),
  orders.region        AS region        WITH SYNONYMS = ('geography', 'location'),
  orders.risk_level    AS risk_level    WITH SYNONYMS = ('risk', 'risk tier', 'risk classification'),
  orders.order_date    AS order_date,
  orders.ops_note      AS ops_note      WITH SYNONYMS = ('operational note', 'ai note', 'action item'),
  fed_sensitive.customer_name AS customer_name WITH SYNONYMS = ('customer', 'buyer', 'client'),
  fed_sensitive.credit_card   AS credit_card   WITH SYNONYMS = ('payment method', 'card number')
)
METRICS (
  orders.total_revenue   AS SUM(orders.amount)      WITH SYNONYMS = ('revenue', 'total sales'),
  orders.order_count     AS COUNT(orders.order_id)  WITH SYNONYMS = ('number of orders'),
  orders.avg_order_value AS AVG(orders.amount)       WITH SYNONYMS = ('average order', 'AOV')
);

In [ ]:
%%sql -r grant_access
GRANT SELECT ON TABLE IDENTIFIER($AI_INSIGHTS_TBL) TO ROLE IDENTIFIER($SF_READER_ROLE);
GRANT SELECT ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW) TO ROLE IDENTIFIER($SF_READER_ROLE);

## Step 5 — Verify the Pipeline Results

In [ ]:
%%sql -r verify_results
-- Verify: Show enriched data with AI-generated ops notes
SELECT order_id, product, amount, status, region, risk_level, ops_note
FROM IDENTIFIER($AI_INSIGHTS_TBL)
ORDER BY amount DESC;

In [ ]:
%%sql -r governance_check
-- Verify governance: risk_level should be masked for non-SYSADMIN roles
SELECT CURRENT_ROLE() AS current_role, 'risk_level is visible because you are SYSADMIN' AS note;